In [7]:
from classes.GaloisField import *

In [8]:
m_order = 4
primitive_poly = 0b11001

gf = GaloisField(m_order, primitive_poly)

print(gf)

gf(17)
gf(ELEMENT_ZERO)
gf(-17)

gf.add(ELEMENT_ZERO,ELEMENT_ZERO)
gf.add(3,3)
gf.add(13,3)
gf.mul(ELEMENT_ZERO,14)
gf.mul(14,ELEMENT_ZERO)
gf.mul(5,3)
gf.inv(20)
# gf.div(0,ELEMENT_ZERO)
gf.div(ELEMENT_ZERO,0)
gf.div(15,7)
gf.div(7,15)
gf.div(4,4)
gf.div(0,4)
gf.pow(ELEMENT_ZERO,2)
gf.pow(222,ELEMENT_ZERO)
gf.pow(3,5)

gf.add(2,15)

(gf.add(gf(2)[0],gf(15)[0]),gf(2)[0],gf(15)[0])



Field Closed Succesfully!, 15 Non-Zero Elements
0001 -- 0
0010 -- 1
0100 -- 2
1000 -- 3
1001 -- 4
1011 -- 5
1111 -- 6
0111 -- 7
1110 -- 8
0101 -- 9
1010 -- 10
1101 -- 11
0011 -- 12
0110 -- 13
1100 -- 14
0000 -- -1



((11, 5), 4, 1)

In [9]:
import numpy as np

In [25]:
def align_terms(f, g):
    diff = len(f) - len(g)
    if diff > 0:
        g = [ELEMENT_ZERO]*diff + g
    elif diff < 0:
        f = [ELEMENT_ZERO]*(-diff) + f

    return f, g

def add_terms(terms_array):
    terms_array = np.atleast_2d(terms_array)
    term, power = terms_array.shape

    ans = []
    for p in range(power):
        sum = ELEMENT_ZERO
        for t in range(term):
            sum = gf.add(terms_array[t,p], sum)[ID_POWER]

        ans += [sum]

    return ans

def add_poly(f, g):
    # complete poly terms
    f, g = align_terms(f, g)

    # add same power terms
    terms_array = np.vstack((f,g))
    ans = add_terms(terms_array)

    return ans

def get_poly_degree(p):
    poly = np.array(p[::-1])
    return poly.size-1

def scale_poly(p, alpha):
    for i, ak in enumerate(p):
        if ak != ELEMENT_ZERO:
            p[i] = gf.mul(ak, alpha)[ID_POWER]
    return p

def mul_poly(f, g):
    # identify longest poly
    f_degree = get_poly_degree(f)
    g_degree = get_poly_degree(g)

    # expand size to match f*g final poly
    if g_degree > f_degree:
        long_poly   = np.array(g[::-1] + [ELEMENT_ZERO]*f_degree)
        short_poly  = np.array(f[::-1])
    else:
        long_poly   = np.array(f[::-1] + [ELEMENT_ZERO]*g_degree)
        short_poly  = np.array(g[::-1])

    term_stack = None
    for i, ak in enumerate(short_poly):
        if ak != ELEMENT_ZERO:
            # manage powers
            rotated = np.roll(long_poly, i)
            # multiply by constant
            rotated     = scale_poly(rotated, ak)
            term_stack  = np.array(rotated) if term_stack is None else np.vstack((term_stack, rotated))

    ans = add_terms(term_stack)

    return ans

def eval_poly(p, x):
    p = np.array(f[::-1])

    for i, ak in enumerate(p):
        if i == 0:
            ans = p[0]
        elif ak != ELEMENT_ZERO:
            # compute powers, multiply by ak, add all
            x_pow   = gf.pow(x, i)[ID_POWER]
            term    = gf.mul(ak, x_pow)[ID_POWER]
            ans     = gf.add(term, ans)[ID_POWER]

    return ans


In [30]:
f = [ELEMENT_ONE, 5, 9]
g = [3, 11, 6]

mul_poly(f,g)[::-1]

eval_poly(f, ELEMENT_ONE)

# gf.add(ELEMENT_ONE,gf.add(5,9)[ID_POWER])


6